# CIFAR-2 / ResNet-9 — LDS Benchmark (Colab GPU)

Compares **Traceprop** vs **TRAK (official, multi-checkpoint)** vs **Random baseline**.

**Runtime:** Set Runtime > Change runtime type > **T4 GPU**. Expected: ~1-2 hours.

TRAK uses the official `traker` library with 5 checkpoints (epochs 16-20), matching the methodology from Park et al. (2023).

Results saved to `cifar2_resnet9_lds.json` — download from Files panel.

In [1]:
!pip install -q traker[fast]

import torch
print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    raise RuntimeError("No GPU! Set Runtime > Change runtime type > T4 GPU")

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
PyTorch 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


## 0. Mount Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

CKPT_DIR = "/content/drive/MyDrive/traceprop_cifar2_ckpts"
import os
os.makedirs(CKPT_DIR, exist_ok=True)
print(f"Checkpoint dir: {CKPT_DIR}")


Mounted at /content/drive
Checkpoint dir: /content/drive/MyDrive/traceprop_cifar2_ckpts


In [3]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Subset
import torchvision
import torchvision.transforms as transforms
from scipy.stats import spearmanr
import json
import time
import os

DEVICE = torch.device("cuda")
# CKPT_DIR is set in the Google Drive cell above
os.makedirs(CKPT_DIR, exist_ok=True)

# --- Config ---
PROJ_DIM = 4096
N_SUBSETS = 500
SUBSET_RATIO = 0.5
LR = 0.1
EPOCHS = 20
BATCH_SIZE = 128
SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)

def ckpt_exists(name):
    return os.path.exists(f"{CKPT_DIR}/{name}")

def save_ckpt(name, **kwargs):
    torch.save(kwargs, f"{CKPT_DIR}/{name}")
    print(f"  [Checkpoint saved: {name}]")

def load_ckpt(name):
    data = torch.load(f"{CKPT_DIR}/{name}", weights_only=False)
    print(f"  [Checkpoint loaded: {name}]")
    return data

## 1. CIFAR-2 Dataset (airplane vs automobile)

In [4]:
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])

full_train = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_test)
full_test = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)

train_indices = [i for i in range(len(full_train)) if full_train.targets[i] in (0, 1)]
test_indices = [i for i in range(len(full_test)) if full_test.targets[i] in (0, 1)]

train_subset = Subset(full_train, train_indices)
test_subset = Subset(full_test, test_indices)

N_TRAIN = len(train_indices)
N_TEST = len(test_indices)
print(f"CIFAR-2: {N_TRAIN} train, {N_TEST} test")

def preload(subset):
    xs, ys = [], []
    for x, y in subset:
        xs.append(x)
        ys.append(y)
    return torch.stack(xs), torch.tensor(ys)

X_train_all, y_train_all = preload(train_subset)
X_test_all, y_test_all = preload(test_subset)
print(f"X_train: {X_train_all.shape}, X_test: {X_test_all.shape}")

100%|██████████| 170M/170M [02:48<00:00, 1.01MB/s]


CIFAR-2: 10000 train, 2000 test
X_train: torch.Size([10000, 3, 32, 32]), X_test: torch.Size([2000, 3, 32, 32])


## 2. ResNet-9 Model

In [5]:
def conv_bn(in_c, out_c, kernel_size=3, stride=1, padding=1):
    return nn.Sequential(
        nn.Conv2d(in_c, out_c, kernel_size, stride, padding, bias=False),
        nn.BatchNorm2d(out_c),
        nn.ReLU(inplace=True),
    )

class ResNet9(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.prep = conv_bn(3, 64)
        self.layer1 = nn.Sequential(conv_bn(64, 128), nn.MaxPool2d(2))
        self.res1 = nn.Sequential(conv_bn(128, 128), conv_bn(128, 128))
        self.layer2 = nn.Sequential(conv_bn(128, 256), nn.MaxPool2d(2))
        self.layer3 = nn.Sequential(conv_bn(256, 512), nn.MaxPool2d(2))
        self.res2 = nn.Sequential(conv_bn(512, 512), conv_bn(512, 512))
        self.pool = nn.AdaptiveMaxPool2d(1)
        self.classifier = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.prep(x)
        x = self.layer1(x)
        x = x + self.res1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = x + self.res2(x)
        x = self.pool(x).flatten(1)
        return self.classifier(x)

# Checkpoint epochs for TRAK multi-checkpoint (last 5 epochs)
CKPT_EPOCHS = [15, 16, 17, 18, 19]  # 0-indexed, so epochs 16-20

def train_resnet(model, X, y, lr=LR, epochs=EPOCHS, batch_size=BATCH_SIZE, save_ckpts=None):
    """Train ResNet-9. If save_ckpts is a list, save state_dict at those epoch indices."""
    model.train()
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    scheduler = optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr,
        steps_per_epoch=(len(X) // batch_size + 1),
        epochs=epochs
    )
    loss_fn = nn.CrossEntropyLoss()
    loader = DataLoader(TensorDataset(X, y), batch_size=batch_size, shuffle=True)
    checkpoints = {}
    for epoch in range(epochs):
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()
            scheduler.step()
        if save_ckpts is not None and epoch in save_ckpts:
            checkpoints[epoch] = {k: v.clone() for k, v in model.state_dict().items()}
    return model, checkpoints

@torch.no_grad()
def get_accuracy(model, X, y, batch_size=512):
    model.eval()
    correct = 0
    for xb, yb in DataLoader(TensorDataset(X, y), batch_size=batch_size):
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        correct += (model(xb).argmax(1) == yb).sum().item()
    return correct / len(X)

@torch.no_grad()
def get_per_sample_correct(model, X, y, batch_size=512):
    """Return binary correctness per test sample."""
    model.eval()
    results = []
    for xb, yb in DataLoader(TensorDataset(X, y), batch_size=batch_size):
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        results.append((model(xb).argmax(1) == yb).cpu().numpy())
    return np.concatenate(results).astype(np.float64)

@torch.no_grad()
def get_per_sample_margin(model, X, y, batch_size=512):
    """Return output margin (logit_correct - logit_incorrect) per test sample.
    This is what the TRAK paper uses for LDS — continuous, more informative than binary."""
    model.eval()
    margins = []
    for xb, yb in DataLoader(TensorDataset(X, y), batch_size=batch_size):
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        logits = model(xb)  # (batch, 2)
        # For binary classification: margin = logit[correct] - logit[incorrect]
        correct_logit = logits.gather(1, yb.unsqueeze(1)).squeeze(1)
        # Get the other logit
        wrong_logit = logits.sum(1) - correct_logit  # works for 2 classes
        margin = (correct_logit - wrong_logit).cpu().numpy()
        margins.append(margin)
    return np.concatenate(margins).astype(np.float64)

n_params = sum(p.numel() for p in ResNet9().parameters())
print(f"ResNet-9 parameters: {n_params:,}")

ResNet-9 parameters: 6,569,026


## 3. Train Full Model

In [6]:
if ckpt_exists("model_multi.pt"):
    ckpt = load_ckpt("model_multi.pt")
    model = ResNet9(num_classes=2).to(DEVICE)
    model.load_state_dict(ckpt["final_state_dict"])
    full_acc = ckpt["full_acc"]
    train_time = ckpt["train_time"]
    multi_ckpts = ckpt["multi_ckpts"]  # dict: epoch -> state_dict
    print(f"Loaded {len(multi_ckpts)} checkpoints from epochs {sorted(multi_ckpts.keys())}")
else:
    torch.manual_seed(SEED)
    model = ResNet9(num_classes=2).to(DEVICE)
    t0 = time.perf_counter()
    model, multi_ckpts = train_resnet(model, X_train_all, y_train_all, save_ckpts=CKPT_EPOCHS)
    train_time = time.perf_counter() - t0
    full_acc = get_accuracy(model, X_test_all, y_test_all)
    save_ckpt("model_multi.pt",
              final_state_dict=model.state_dict(),
              multi_ckpts=multi_ckpts,
              full_acc=full_acc, train_time=train_time)
    print(f"Saved {len(multi_ckpts)} checkpoints from epochs {sorted(multi_ckpts.keys())}")

print(f"Full model accuracy: {full_acc:.4f} (trained in {train_time:.1f}s)")
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params: {n_params:,}")

  [Checkpoint loaded: model_multi.pt]
Loaded 5 checkpoints from epochs [15, 16, 17, 18, 19]
Full model accuracy: 0.9735 (trained in 80.0s)
Trainable params: 6,569,026


## 4. TRAK Attribution (official library, multi-checkpoint)

In [7]:
if ckpt_exists("trak_influence.pt"):
    ckpt = load_ckpt("trak_influence.pt")
    trak_influence_matrix = ckpt["trak_influence_matrix"]
    trak_time = ckpt["trak_time"]
    print(f"TRAK loaded: {trak_influence_matrix.shape}, took {trak_time:.1f}s originally")
else:
    from trak import TRAKer

    print(f"Computing TRAK attribution with {len(multi_ckpts)} checkpoints...")
    t0 = time.perf_counter()

    traker = TRAKer(model=ResNet9(num_classes=2).to(DEVICE),
                    task='image_classification',
                    train_set_size=N_TRAIN,
                    proj_dim=PROJ_DIM,
                    save_dir='./trak_cache',
                    device=DEVICE,
                    proj_max_batch_size=8,
                    use_half_precision=False)

    # Featurize training data for each checkpoint
    loader_train = DataLoader(TensorDataset(X_train_all, y_train_all),
                              batch_size=16, shuffle=False)
    for model_id, epoch in enumerate(sorted(multi_ckpts.keys())):
        sd = multi_ckpts[epoch]
        traker.load_checkpoint(sd, model_id=model_id)
        for xb, yb in loader_train:
            traker.featurize(batch=(xb.to(DEVICE), yb.to(DEVICE)),
                           num_samples=xb.shape[0])
        print(f"  Featurized checkpoint {model_id} (epoch {epoch+1})")
    traker.finalize_features()
    print("  Features finalized")

    # Score test data for each checkpoint
    loader_test = DataLoader(TensorDataset(X_test_all, y_test_all),
                             batch_size=16, shuffle=False)
    for model_id, epoch in enumerate(sorted(multi_ckpts.keys())):
        sd = multi_ckpts[epoch]
        traker.start_scoring_checkpoint(exp_name='lds',
                                        checkpoint=sd,
                                        model_id=model_id,
                                        num_targets=N_TEST)
        for xb, yb in loader_test:
            traker.score(batch=(xb.to(DEVICE), yb.to(DEVICE)),
                        num_samples=xb.shape[0])
        print(f"  Scored checkpoint {model_id} (epoch {epoch+1})")

    trak_scores = traker.finalize_scores(exp_name='lds')
    trak_influence_matrix = np.array(trak_scores)  # shape: (N_TEST, N_TRAIN)
    trak_time = time.perf_counter() - t0
    save_ckpt("trak_influence.pt", trak_influence_matrix=trak_influence_matrix, trak_time=trak_time)
    print(f"TRAK done: {trak_influence_matrix.shape}, {trak_time:.1f}s")

  [Checkpoint loaded: trak_influence.pt]
TRAK loaded: (10000, 2000), took 691.3s originally


## 5. Traceprop Attribution — Two Variants

### Variant A: Batch-mean (current library behaviour)
One gradient per batch over ALL parameters → assigned to every sample in that batch.

### Variant B: Last-layer per-sample (new)
Captures the exact per-sample gradient of the **final linear layer only** (512×2 = 1024 floats).  
Uses the identity: `∂CE/∂W = (softmax(logits) - one_hot(y)) ⊗ feature`.  
No backward-pass loop, no OOM, exact per-sample signal.

In [8]:
if ckpt_exists("tp_influence.pt"):
    ckpt = load_ckpt("tp_influence.pt")
    tp_influence_matrix = ckpt["tp_influence_matrix"]
    tp_time = ckpt["tp_time"]
    tp_n_ckpts = 1
    print(f"Traceprop loaded: {tp_influence_matrix.shape}, took {tp_time:.1f}s originally")
else:
    print("Computing Traceprop attribution (batch-mean, count-sketch JL, single checkpoint)...")
    t0 = time.perf_counter()

    torch.manual_seed(42)
    tp_hash_buckets = torch.randint(0, PROJ_DIM, (n_params,), device=DEVICE)
    tp_hash_signs = (torch.randint(0, 2, (n_params,), device=DEVICE).float() * 2 - 1) * (1.0 / PROJ_DIM) ** 0.5

    def tp_project(full_grad):
        proj = torch.zeros(PROJ_DIM, device=DEVICE)
        proj.scatter_add_(0, tp_hash_buckets, full_grad * tp_hash_signs)
        return proj

    model.eval()
    loss_fn = nn.CrossEntropyLoss()

    # Batch-mean gradients (matching TrainingContext.step behavior)
    loader = DataLoader(TensorDataset(X_train_all, y_train_all), batch_size=BATCH_SIZE, shuffle=False)
    batch_projs = []
    batch_sizes = []
    for batch_i, (xb, yb) in enumerate(loader):
        model.zero_grad()
        out = model(xb.to(DEVICE))
        loss = loss_fn(out, yb.to(DEVICE))
        loss.backward()
        full_grad = torch.cat([p.grad.flatten() for p in model.parameters() if p.requires_grad])
        batch_projs.append(tp_project(full_grad))
        batch_sizes.append(xb.shape[0])
        if batch_i % 20 == 0:
            print(f"  Train batch {batch_i}/{len(loader)} ({time.perf_counter()-t0:.0f}s)")

    tp_train_projs = torch.zeros(N_TRAIN, PROJ_DIM, device=DEVICE)
    idx = 0
    for proj, bs in zip(batch_projs, batch_sizes):
        tp_train_projs[idx:idx+bs] = proj.unsqueeze(0)
        idx += bs

    # Test gradients (per-sample)
    tp_test_projs = torch.zeros(N_TEST, PROJ_DIM, device=DEVICE)
    for i in range(N_TEST):
        model.zero_grad()
        out = model(X_test_all[i].unsqueeze(0).to(DEVICE))
        loss = loss_fn(out, y_test_all[i].unsqueeze(0).to(DEVICE))
        loss.backward()
        full_grad = torch.cat([p.grad.flatten() for p in model.parameters() if p.requires_grad])
        tp_test_projs[i] = tp_project(full_grad)
        if i % 500 == 0:
            print(f"  Test {i}/{N_TEST} ({time.perf_counter()-t0:.0f}s)")

    tp_influence_matrix = (tp_test_projs @ tp_train_projs.T).cpu().numpy()
    tp_time = time.perf_counter() - t0
    tp_n_ckpts = 1
    save_ckpt("tp_influence.pt", tp_influence_matrix=tp_influence_matrix, tp_time=tp_time)
    print(f"Traceprop done: {tp_influence_matrix.shape}, {tp_time:.1f}s")

  [Checkpoint loaded: tp_influence.pt]
Traceprop loaded: (2000, 10000), took 14.2s originally


In [9]:
# ── Variant B: Last-layer per-sample gradients (no projection needed — 1024 dims is small) ──
if ckpt_exists("tp_ll_influence.pt"):
    ckpt = load_ckpt("tp_ll_influence.pt")
    tp_ll_influence_matrix = ckpt["tp_ll_influence_matrix"]
    tp_ll_time = ckpt["tp_ll_time"]
    print(f"Traceprop-LL loaded: {tp_ll_influence_matrix.shape}, took {tp_ll_time:.1f}s originally")
else:
    import torch.nn.functional as F

    print("Computing Traceprop-LL attribution (last-layer per-sample, raw 1024-dim)...")
    t0 = time.perf_counter()

    LL_DIM = 2 * 512  # = 1024 — last linear layer gradient dim (no projection needed)

    # Hook to capture features entering the classifier (shape: batch × 512)
    captured = {}
    def feat_hook(module, inp, out):
        captured['feat'] = inp[0].detach()
    hook_handle = model.classifier.register_forward_hook(feat_hook)

    model.eval()

    # ── Training set ──────────────────────────────────────────────────────────
    LL_BATCH = 128
    train_loader_ll = DataLoader(TensorDataset(X_train_all, y_train_all),
                                 batch_size=LL_BATCH, shuffle=False)
    tp_ll_train = torch.zeros(N_TRAIN, LL_DIM, device=DEVICE)
    idx = 0
    for bi, (xb, yb) in enumerate(train_loader_ll):
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        with torch.no_grad():
            logits = model(xb)
            feat = captured['feat']            # (B, 512)
            sm = F.softmax(logits, dim=1)      # (B, 2)
            oh = F.one_hot(yb, num_classes=2).float()
            output_grad = sm - oh              # (B, 2) — exact ∂CE/∂logits
        # per-sample ∂CE/∂W = output_grad_i ⊗ feat_i → flatten to (B, 1024)
        per_sample = torch.einsum('bi,bj->bij', output_grad, feat).reshape(xb.shape[0], -1)
        tp_ll_train[idx:idx + xb.shape[0]] = per_sample
        idx += xb.shape[0]
        if bi % 30 == 0:
            print(f"  Train batch {bi}/{len(train_loader_ll)} ({time.perf_counter()-t0:.0f}s)")

    # ── Test set ──────────────────────────────────────────────────────────────
    test_loader_ll = DataLoader(TensorDataset(X_test_all, y_test_all),
                                batch_size=LL_BATCH, shuffle=False)
    tp_ll_test = torch.zeros(N_TEST, LL_DIM, device=DEVICE)
    idx = 0
    for bi, (xb, yb) in enumerate(test_loader_ll):
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        with torch.no_grad():
            logits = model(xb)
            feat = captured['feat']
            sm = F.softmax(logits, dim=1)
            oh = F.one_hot(yb, num_classes=2).float()
            output_grad = sm - oh
        per_sample = torch.einsum('bi,bj->bij', output_grad, feat).reshape(xb.shape[0], -1)
        tp_ll_test[idx:idx + xb.shape[0]] = per_sample
        idx += xb.shape[0]

    hook_handle.remove()

    tp_ll_influence_matrix = (tp_ll_test @ tp_ll_train.T).cpu().numpy()
    tp_ll_time = time.perf_counter() - t0
    save_ckpt("tp_ll_influence.pt",
              tp_ll_influence_matrix=tp_ll_influence_matrix,
              tp_ll_time=tp_ll_time)
    print(f"Traceprop-LL done: {tp_ll_influence_matrix.shape}, {tp_ll_time:.1f}s")

Computing Traceprop-LL attribution (last-layer per-sample, raw 1024-dim)...
  Train batch 0/79 (1s)
  Train batch 30/79 (1s)
  Train batch 60/79 (2s)
  [Checkpoint saved: tp_ll_influence.pt]
Traceprop-LL done: (2000, 10000), 2.6s


## 6. Ground-Truth Retraining (50 subsets)

In [10]:
print(f"Retraining {N_SUBSETS} models for ground truth LDS...")
t0 = time.perf_counter()

np.random.seed(SEED + 1000)
all_masks = [np.random.rand(N_TRAIN) < SUBSET_RATIO for _ in range(N_SUBSETS)]

# Resume from checkpoint if available
if ckpt_exists("retrain_margin.pt"):
    ckpt = load_ckpt("retrain_margin.pt")
    subset_masks = ckpt["subset_masks"]
    subset_margins = ckpt["subset_margins"]
    start_s = int(ckpt["completed"])
    print(f"  Resuming from subset {start_s}/{N_SUBSETS}")
else:
    subset_masks = np.array(all_masks)
    subset_margins = np.zeros((N_SUBSETS, N_TEST))
    start_s = 0

for s in range(start_s, N_SUBSETS):
    mask = all_masks[s]
    torch.manual_seed(s)
    m = ResNet9(num_classes=2).to(DEVICE)
    m, _ = train_resnet(m, X_train_all[mask], y_train_all[mask])
    subset_margins[s] = get_per_sample_margin(m, X_test_all, y_test_all)

    acc = (subset_margins[s] > 0).mean()
    elapsed = time.perf_counter() - t0
    done = s - start_s + 1
    eta = elapsed / done * (N_SUBSETS - s - 1)
    print(f"  Subset {s+1}/{N_SUBSETS}: acc={acc:.4f} [{elapsed/60:.0f}m elapsed, ~{eta/60:.0f}m remaining]")

    save_ckpt("retrain_margin.pt",
              subset_masks=subset_masks, subset_margins=subset_margins, completed=s+1)

retrain_time = time.perf_counter() - t0
print(f"\nRetraining took {retrain_time/60:.1f} min")

Retraining 500 models for ground truth LDS...
  [Checkpoint loaded: retrain_margin.pt]
  Resuming from subset 500/500

Retraining took 0.0 min


In [11]:
import warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)

# Fix TRAK matrix orientation if needed
if trak_influence_matrix.shape == (N_TRAIN, N_TEST):
    trak_influence_matrix = trak_influence_matrix.T
    print(f"Transposed TRAK matrix to {trak_influence_matrix.shape}")

def compute_lds(influence_matrix, subset_masks, subset_outcomes, n_test, label=""):
    lds_scores = []
    for test_i in range(n_test):
        predicted = subset_masks @ influence_matrix[test_i]
        actual = subset_outcomes[:, test_i]
        corr = spearmanr(predicted, actual).statistic
        lds_scores.append(corr if not np.isnan(corr) else 0.0)
        if test_i % 200 == 0 and label:
            print(f"  [{label}] Test point {test_i}/{n_test}")
    return np.array(lds_scores)

print("Computing LDS (margin-based)...")

print("\nTRAK:")
lds_trak = compute_lds(trak_influence_matrix, subset_masks, subset_margins, N_TEST, "TRAK")
print(f"  TRAK LDS: {lds_trak.mean():.4f} +/- {lds_trak.std():.4f}")

print("\nTraceprop (batch-mean):")
lds_tp = compute_lds(tp_influence_matrix, subset_masks, subset_margins, N_TEST, "TP-batchmean")
print(f"  Traceprop-BM LDS: {lds_tp.mean():.4f} +/- {lds_tp.std():.4f}")

print("\nTraceprop-LL (last-layer per-sample):")
lds_tp_ll = compute_lds(tp_ll_influence_matrix, subset_masks, subset_margins, N_TEST, "TP-LL")
print(f"  Traceprop-LL LDS: {lds_tp_ll.mean():.4f} +/- {lds_tp_ll.std():.4f}")

print("\nRandom:")
np.random.seed(SEED + 2000)
random_matrix = np.random.rand(N_TEST, N_TRAIN)
lds_rand = compute_lds(random_matrix, subset_masks, subset_margins, N_TEST)
print(f"  Random LDS: {lds_rand.mean():.4f} +/- {lds_rand.std():.4f}")

Transposed TRAK matrix to (2000, 10000)
Computing LDS (margin-based)...

TRAK:
  [TRAK] Test point 0/2000
  [TRAK] Test point 200/2000
  [TRAK] Test point 400/2000
  [TRAK] Test point 600/2000
  [TRAK] Test point 800/2000
  [TRAK] Test point 1000/2000
  [TRAK] Test point 1200/2000
  [TRAK] Test point 1400/2000
  [TRAK] Test point 1600/2000
  [TRAK] Test point 1800/2000
  TRAK LDS: 0.0290 +/- 0.0523

Traceprop (batch-mean):
  [TP-batchmean] Test point 0/2000
  [TP-batchmean] Test point 200/2000
  [TP-batchmean] Test point 400/2000
  [TP-batchmean] Test point 600/2000
  [TP-batchmean] Test point 800/2000
  [TP-batchmean] Test point 1000/2000
  [TP-batchmean] Test point 1200/2000
  [TP-batchmean] Test point 1400/2000
  [TP-batchmean] Test point 1600/2000
  [TP-batchmean] Test point 1800/2000
  Traceprop-BM LDS: 0.0033 +/- 0.0334

Traceprop-LL (last-layer per-sample):
  [TP-LL] Test point 0/2000
  [TP-LL] Test point 200/2000
  [TP-LL] Test point 400/2000
  [TP-LL] Test point 600/2000
  [TP

In [12]:
print("\n" + "=" * 65)
print("CIFAR-2 / ResNet-9 — LDS Results")
print("=" * 65)
print(f'{"Method":<40} {"LDS mean":>8} {"± std":>8} {"Time":>8}')
print("-" * 65)
print(f'{"TRAK (5 ckpts, official)":<40} {lds_trak.mean():>8.4f} {lds_trak.std():>8.4f} {trak_time:>7.1f}s')
print(f'{"Traceprop-BM (batch-mean, 1 ckpt)":<40} {lds_tp.mean():>8.4f} {lds_tp.std():>8.4f} {tp_time:>7.1f}s')
print(f'{"Traceprop-LL (last-layer per-sample)":<40} {lds_tp_ll.mean():>8.4f} {lds_tp_ll.std():>8.4f} {tp_ll_time:>7.1f}s')
print(f'{"Random baseline":<40} {lds_rand.mean():>8.4f} {lds_rand.std():>8.4f} {"<0.001s":>8}')
print("=" * 65)

results = {
    "benchmark": "CIFAR-2/ResNet-9",
    "n_train": N_TRAIN,
    "n_test": N_TEST,
    "n_subsets": N_SUBSETS,
    "proj_dim": PROJ_DIM,
    "epochs": EPOCHS,
    "full_model_accuracy": round(full_acc, 4),
    "trak": {
        "lds_mean": round(float(lds_trak.mean()), 4),
        "lds_std": round(float(lds_trak.std()), 4),
        "time_s": round(trak_time, 1),
        "method": "official_traker_5_checkpoints",
    },
    "traceprop_batchmean": {
        "lds_mean": round(float(lds_tp.mean()), 4),
        "lds_std": round(float(lds_tp.std()), 4),
        "time_s": round(tp_time, 1),
        "method": "batch_mean_all_params_single_checkpoint",
    },
    "traceprop_lastlayer": {
        "lds_mean": round(float(lds_tp_ll.mean()), 4),
        "lds_std": round(float(lds_tp_ll.std()), 4),
        "time_s": round(tp_ll_time, 1),
        "method": "per_sample_last_layer_512x2_single_checkpoint",
    },
    "random": {
        "lds_mean": round(float(lds_rand.mean()), 4),
        "lds_std": round(float(lds_rand.std()), 4),
    },
    "retrain_time_s": round(retrain_time, 1),
    "device": str(DEVICE),
}

with open("cifar2_resnet9_lds.json", "w") as f:
    json.dump(results, f, indent=2)
print("\nSaved to cifar2_resnet9_lds.json")
print(json.dumps(results, indent=2))


CIFAR-2 / ResNet-9 — LDS Results
Method                                   LDS mean    ± std     Time
-----------------------------------------------------------------
TRAK (5 ckpts, official)                   0.0290   0.0523   691.3s
Traceprop-BM (batch-mean, 1 ckpt)          0.0033   0.0334    14.2s
Traceprop-LL (last-layer per-sample)       0.0168   0.0684     2.6s
Random baseline                            0.0205   0.0357  <0.001s

Saved to cifar2_resnet9_lds.json
{
  "benchmark": "CIFAR-2/ResNet-9",
  "n_train": 10000,
  "n_test": 2000,
  "n_subsets": 500,
  "proj_dim": 4096,
  "epochs": 20,
  "full_model_accuracy": 0.9735,
  "trak": {
    "lds_mean": 0.029,
    "lds_std": 0.0523,
    "time_s": 691.3,
    "method": "official_traker_5_checkpoints"
  },
  "traceprop_batchmean": {
    "lds_mean": 0.0033,
    "lds_std": 0.0334,
    "time_s": 14.2,
    "method": "batch_mean_all_params_single_checkpoint"
  },
  "traceprop_lastlayer": {
    "lds_mean": 0.0168,
    "lds_std": 0.0684,
   